# Train the Barangay Buru-un model

This notebook trains the two models the app uses:

- **Decision tree** — a short list of yes/no questions that ends at HIGH, MEDIUM, or LOW.
- **Logistic regression** — one weight per household feature. A positive weight for HIGH pushes that household toward high need.

The score shown in the app is a **50/50 blend** of the two models' probabilities.

Training uses the **483 real Data Sets members** (name, position, association). Age, income, family size, housing, and the label are still estimated, because the sheet leaves those blank. No generated households are added.

A CPU runtime is enough. Do not turn on a GPU.

## What to do

1. Open [Google Colab](https://colab.research.google.com).
2. **File → Upload notebook** and choose this file.
3. **Runtime → Change runtime type → CPU → Save.**
4. **Runtime → Run all.**
5. When the second cell asks, upload these files from the `ml-service` folder on your computer:

   - `features.py`
   - `dataset.py`
   - `models.py`
   - `real_data.py`
   - `train.py`

   Skip the upload if those files are already in the same folder as this notebook.
6. Wait until the training cell prints `PASS` or `FAIL`. The target is **80% blended accuracy** on the held-out 25%.
7. The last cell downloads `vulnerability_model.joblib` and `metrics.json`.
8. Put both files in `ml-service/artifacts/` on your computer (create the folder if it is missing).
9. Restart the app (`npm run dev`). The ML service loads those two files on startup.


In [ ]:
# Install the same libraries the Flask service uses.
%pip install -q "numpy>=2.0.0" "scikit-learn>=1.6.0" "joblib>=1.4.0" pandas matplotlib


In [ ]:
import sys
from pathlib import Path

NEEDED = ["features.py", "dataset.py", "models.py", "real_data.py", "train.py"]

def find_code_dir():
    here = Path.cwd()
    for candidate in (here, here / "ml-service", Path("/content")):
        if all((candidate / name).exists() for name in NEEDED):
            return candidate
    return None

code_dir = find_code_dir()
if code_dir is None:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit(
            "Run this notebook from the ml-service folder, or open it in Google Colab so you can upload the five Python files."
        )
    print("Select these five files from the ml-service folder:")
    for name in NEEDED:
        print("  ", name)
    uploaded = files.upload()
    missing = [name for name in NEEDED if name not in uploaded]
    if missing:
        raise SystemExit("Still missing: " + ", ".join(missing))
    code_dir = Path.cwd()

if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))
print("Using code from", code_dir)


In [ ]:
from train import train_with_real_records, TARGET_ACCURACY, CV_FOLDS

scorer, metrics, summary = train_with_real_records()
cv = metrics["crossValidation"]["blended"]

print(f"members trained on     : {summary['realRosterRows']}")
print(f"generated households   : {summary['nSynthetic']}")
print(f"train / test           : {summary['trainSize']} / {summary['testSize']}")
print(f"decision tree          : {metrics['decisionTree']['accuracy']:.4f}")
print(f"logistic regression    : {metrics['logisticRegression']['accuracy']:.4f}")
print(f"blended (what the app shows) : {metrics['blended']['accuracy']:.4f}")
print(f"{CV_FOLDS}-fold blended average : {cv['mean']:.4f} +/- {cv['std']:.4f}")
print(f"target >= {TARGET_ACCURACY:.2f}  ->  {'PASS' if summary['meetsTarget'] else 'FAIL'}")
print()
print("Associations:")
for name, count in summary["roster"]["associations"].items():
    print(f"  {count:3}  {name}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

importance = pd.Series(
    {row["feature"]: row["importance"] for row in scorer.feature_importance()}
).sort_values()
coefficients = pd.Series(
    {row["feature"]: row["coefficient"] for row in scorer.coefficients()}
).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
importance.plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Decision tree — which features the questions use")
coefficients.plot.barh(ax=axes[1], color="darkorange")
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_title("Logistic regression — weight toward HIGH need")
plt.tight_layout()
plt.show()

print("Decision tree rules (first 8):")
for rule in scorer.tree_rules()[:8]:
    print(" ", rule)


In [ ]:
import json
import os
from datetime import datetime, timezone

import joblib
from features import FEATURE_NAMES

os.makedirs("artifacts", exist_ok=True)
model_path = os.path.join("artifacts", "vulnerability_model.joblib")
metrics_path = os.path.join("artifacts", "metrics.json")

# Same file shape app.py loads: the key must be "scorer".
joblib.dump({"scorer": scorer, "featureNames": list(FEATURE_NAMES)}, model_path)
trained_at = datetime.now(timezone.utc).isoformat()
with open(metrics_path, "w", encoding="utf-8") as handle:
    json.dump(
        {"metrics": metrics, "summary": summary, "trainedAt": trained_at},
        handle,
        indent=2,
    )

print("Saved")
print(" ", model_path)
print(" ", metrics_path)
print("Copy both files into ml-service/artifacts/ and restart npm run dev.")

try:
    from google.colab import files
    files.download(metrics_path)
    files.download(model_path)
except ImportError:
    print("Not running in Colab. The files are already in ./artifacts/")
